In [ ]:
# Configure AWS profile for local development
%env AWS_PROFILE=platform-developer

In [ ]:
import sys
from pathlib import Path

# Add src to sys.path
src_path = Path("..").resolve() / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Select which adapter to use - change this to "folio" or "axiell"
ADAPTER = "axiell"  # Options: "axiell", "folio"
# Separate kill switch for any remote table recreation
ALLOW_REMOTE_TABLE_RECREATION = False

if ADAPTER == "axiell":
    from adapters.axiell import config
    adapter_config = config.AXIELL_ADAPTER_CONFIG
elif ADAPTER == "folio":
    from adapters.folio import config
    adapter_config = config.FOLIO_ADAPTER_CONFIG
else:
    raise ValueError(f"Unknown adapter: {ADAPTER}")

from adapters.utils.iceberg import (
    get_local_catalog_params,
    get_rest_api_catalog_params,
    get_table,
    LocalIcebergTableConfig,
    RestApiIcebergTableConfig,
 )
from pyiceberg.catalog import load_catalog
from pyiceberg.exceptions import NoSuchTableError

print(f"Using {ADAPTER.upper()} adapter")
print("Remote table recreation is blocked unless ALLOW_REMOTE_TABLE_RECREATION=True")

In [ ]:
def recreate_adapter_table(target: str = "local"):
    """Recreate the adapter Iceberg table."""
    if target == "local":
        table_config = adapter_config.local_iceberg_config
        params = get_local_catalog_params(db_name=table_config.db_name)
        catalog_name = "local"
        print(
            f"Recreating LOCAL table: {table_config.namespace}.{table_config.table_name}"
        )
    elif target == "remote":
        if not ALLOW_REMOTE_TABLE_RECREATION:
            raise RuntimeError(
                "Remote adapter table recreation is blocked. "
                "Set ALLOW_REMOTE_TABLE_RECREATION=True only for intentional remote changes."
            )
        table_config = adapter_config.rest_api_iceberg_config
        params = get_rest_api_catalog_params(
            s3_tables_bucket=table_config.s3_tables_bucket,
            region=table_config.region,
            account_id=table_config.account_id,
        )
        catalog_name = "s3tablescatalog"
        print(
            f"Recreating REMOTE table: {table_config.namespace}.{table_config.table_name}"
        )
    else:
        raise ValueError(f"Unknown target: {target}")

    catalog = load_catalog(catalog_name, **params)
    identifier = f"{table_config.namespace}.{table_config.table_name}"

    try:
        print(f"Dropping table {identifier}...")
        if isinstance(table_config, RestApiIcebergTableConfig):
            catalog.drop_table(identifier, purge_requested=True)
        else:
            catalog.drop_table(identifier)
        print("Table dropped.")
    except NoSuchTableError:
        print("Table did not exist.")

    print(f"Creating table {identifier}...")
    get_table(
        config=table_config,
        catalogue_name=catalog_name,
        create_if_not_exists=True,
        **params,
    )
    print("Table created successfully.")

In [ ]:
# Recreate local adapter table
recreate_adapter_table(target="local")

In [ ]:
# Recreate remote adapter table only after explicitly opting in
# ALLOW_REMOTE_TABLE_RECREATION = True
# recreate_adapter_table(target="remote")

In [ ]:
def recreate_window_status_table(target: str = "local"):
    """Recreate the window-status Iceberg table."""
    if target == "local":
        table_config = adapter_config.local_window_status_iceberg_config
        params = get_local_catalog_params(db_name=table_config.db_name)
        catalog_name = "local"
        print(
            "Recreating LOCAL window status table: "
            f"{table_config.namespace}.{table_config.table_name}"
        )
    elif target == "remote":
        if not ALLOW_REMOTE_TABLE_RECREATION:
            raise RuntimeError(
                "Remote window-status table recreation is blocked. "
                "Set ALLOW_REMOTE_TABLE_RECREATION=True only for intentional remote changes."
            )
        table_config = adapter_config.rest_api_window_status_iceberg_config
        params = get_rest_api_catalog_params(
            s3_tables_bucket=table_config.s3_tables_bucket,
            region=table_config.region,
            account_id=table_config.account_id,
        )
        catalog_name = "s3tablescatalog"
        print(
            "Recreating REMOTE window status table: "
            f"{table_config.namespace}.{table_config.table_name}"
        )
    else:
        raise ValueError(f"Unknown target: {target}")

    catalog = load_catalog(catalog_name, **params)
    identifier = f"{table_config.namespace}.{table_config.table_name}"

    try:
        print(f"Dropping table {identifier}...")
        if isinstance(table_config, RestApiIcebergTableConfig):
            catalog.drop_table(identifier, purge_requested=True)
        else:
            catalog.drop_table(identifier)
        print("Table dropped.")
    except NoSuchTableError:
        print("Table did not exist.")

    print(f"Creating table {identifier}...")
    get_table(
        config=table_config,
        catalogue_name=catalog_name,
        create_if_not_exists=True,
        **params,
    )
    print("Table created successfully.")

In [ ]:
# Recreate local window status table
recreate_window_status_table(target="local")

In [ ]:
# Recreate remote window-status table only after explicitly opting in
# ALLOW_REMOTE_TABLE_RECREATION = True
# recreate_window_status_table(target="remote")